# kolm Compile on Colab — free-tier, no local GPU

This notebook shows the **end-to-end distill loop** on Google Colab. It works two ways:

1. **You have a Colab GPU** (T4 / L4 / A100) — the compile runs right here on Colab's GPU.
2. **You don't** — the notebook relays the spec to Modal or RunPod via the kolm cloud route. (You'll need either a paid Kolm plan or your own Modal/RunPod token.)

At the end you'll have a `.kolm` artifact you can verify locally, push to the hub, or serve over OpenAI-compatible HTTP.

**Persona E — Hobbyist / Learner.** No install. No card. No CUDA setup.

## TL;DR

```
captures → spec.json → kolm compile (local Colab GPU OR cloud) → artifact.kolm
```


## 1. What hardware did Colab give us?

Runtime → Change runtime type → T4 GPU (free) or A100 (paid) before running this cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader 2>/dev/null || echo 'no GPU — will use the cloud relay'

## 2. Install kolm

The Python SDK is a thin client. For the full toolchain we also install the Node CLI (Colab has Node 18+ pre-installed).

In [ ]:
# Python SDK (the lightweight wrapper) — pulled from the repo until pip publish lands
!pip install -q 'git+https://github.com/kolm-ai/kolmogorov-stack@main#subdirectory=sdk/python'

# Node CLI (kolm) — needed for `kolm compile`, `kolm verify`, `kolm run`
!npm install -g --silent https://github.com/kolm-ai/kolmogorov-stack.git 2>/dev/null || true
!node -e "console.log('node', process.version)"
!which kolm && kolm --version || echo '(kolm CLI not on PATH yet — use python SDK below)'

## 3. API key

Free signup is one HTTP call. If you already have a key, paste it.

**Never commit your key to a public notebook.** Use Colab → ⚿ Secrets, then read it with `userdata.get('KOLM_API_KEY')`.

In [ ]:
import os, getpass
try:
    from google.colab import userdata  # type: ignore
    KEY = userdata.get('KOLM_API_KEY')
except Exception:
    KEY = None
if not KEY:
    KEY = getpass.getpass('paste your KOLM_API_KEY (ks_... — or press Enter to skip and sign up below): ').strip()
if not KEY:
    import urllib.request, json
    email = input('email for free signup: ').strip()
    req = urllib.request.Request('https://kolm.ai/v1/signup',
                                  data=json.dumps({'email': email}).encode(),
                                  headers={'content-type': 'application/json'})
    with urllib.request.urlopen(req, timeout=15) as r:
        resp = json.loads(r.read())
    KEY = resp.get('api_key')
    print('signed up; tenant', resp.get('tenant_id'), 'key prefix', KEY[:14] + '...')
os.environ['KOLM_API_KEY'] = KEY
os.environ['KOLM_BASE_URL'] = 'https://kolm.ai'
print('using key prefix:', KEY[:14] + '...')

## 4. Author a tiny spec

This is sentiment binary classification — six positives, six negatives. Real specs hit 200-500 examples; this is the smallest possible loop.

In [ ]:
import json, pathlib
spec = {
    'job_id': 'colab_demo',
    'task': 'sentiment binary classifier (Colab demo)',
    'base_model': 'Qwen/Qwen2.5-0.5B-Instruct',  # tiny: fits anywhere
    'examples': 12,
    'epochs': 2,
    'batch_size': 4,
    'seq_len': 256,
    'recipes': [],
    'evals': {
        'spec': 'rs-1-evals',
        'cases': [
            {'id': 'p1', 'input': 'loved this, amazing experience', 'expected': True},
            {'id': 'p2', 'input': 'incredible value, fantastic',     'expected': True},
            {'id': 'p3', 'input': 'great product, very happy',       'expected': True},
            {'id': 'n1', 'input': 'absolutely terrible, worst ever', 'expected': False},
            {'id': 'n2', 'input': 'broken on arrival, hate it',      'expected': False},
            {'id': 'n3', 'input': 'do not buy, awful experience',    'expected': False},
        ],
        'coverage': 1.0,
    },
    'training_stats': {'pass_rate_positive': None, 'latency_p50_us': None},
}
pathlib.Path('colab_demo_spec.json').write_text(json.dumps(spec, indent=2))
print('wrote colab_demo_spec.json,', len(spec['evals']['cases']), 'eval cases')

## 5. Compile

Two paths — the notebook picks one based on whether you have a GPU.

In [ ]:
import subprocess, shutil, os
have_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
have_cli = shutil.which('kolm') is not None
if have_gpu and have_cli:
    print('LOCAL compile path: Colab GPU + kolm CLI')
    print('  this uses the same path as `kolm compile --spec ... ` on your laptop')
    !kolm compile --spec colab_demo_spec.json --no-fit-check
elif have_cli:
    print('CLOUD relay path: no Colab GPU, routing to Modal')
    !kolm compile --spec colab_demo_spec.json --cloud modal --budget 1.00 --confirm
else:
    print('CLOUD relay path (SDK fallback): no CLI, no GPU — using Python SDK')
    import urllib.request, json
    body = {
        'backend': 'modal',
        'spec': json.load(open('colab_demo_spec.json')),
        'budget_usd': 1.00,
        'confirm': True,
    }
    req = urllib.request.Request(
        f"{os.environ['KOLM_BASE_URL']}/v1/compile/cloud",
        data=json.dumps(body).encode(),
        headers={'content-type': 'application/json',
                 'authorization': f"Bearer {os.environ['KOLM_API_KEY']}"},
        method='POST',
    )
    with urllib.request.urlopen(req, timeout=600) as r:
        result = json.loads(r.read())
    print(json.dumps(result, indent=2)[:1200])

## 6. Verify the artifact (no GPU needed)

Verification is pure-CPU — it replays the frozen eval set against the artifact's recipes and prints a K-score line.

In [ ]:
import glob
arts = sorted(glob.glob(os.path.expanduser('~/.kolm/artifacts/*.kolm')), key=os.path.getmtime)
if not arts:
    print('no local artifact (cloud-relay path stores artifacts on Kolm hub).')
    print('list yours with: kolm hub list')
else:
    latest = arts[-1]
    print('verifying', latest)
    !kolm verify {latest}

## 7. What next?

- `kolm run <artifact.kolm> --input 'this product is great'` — one-shot inference, microseconds.
- `kolm serve --http <artifact.kolm> --port 8765` — OpenAI-compatible local server. Hits Colab's GPU automatically; the runtime + dtype are auto-picked from `nvidia-smi`.
- `kolm publish <artifact.kolm> --public` — push to the Kolm hub; anyone can `kolm pull yourname/sentiment-binary`.
- **Studio wizard**: open https://kolm.ai/studio/compile in another tab and walk the same flow with a UI.

## When to stay on Colab vs. graduate to your own box

| Model size | Colab T4 (free, 16GB) | Colab A100 (paid, 40GB) | Cloud relay |
|------------|-----------------------|--------------------------|-------------|
| 0.5B–1.5B  | ✓ fast                | ✓ instant                | ✓ overkill  |
| 3B–7B      | ✓ slow                | ✓ fast                   | ✓ fast      |
| 14B–32B    | ✗ OOM                 | ✓ tight                  | ✓ recommended |
| 70B+       | ✗                     | ✗                        | ✓ only      |
